# NaturalAGI — Quick Start

Цей ноутбук дозволяє запустити повний пайплайн NaturalAGI: скелетизацію, навчання та класифікацію.
Всі функції винесені в окремі модулі — тут лише **конфігурація** та **виклики**.

**Перед запуском:** переконайтесь що сервіси запущені (`make` або `make start_services && make deploy`).

## ⚙️ Конфігурація
Змініть параметри тут — решта ноутбуку їх використає автоматично.

In [ ]:
import os

# ── Шляхи ──────────────────────────────────────────────────────────────────
PROJECT_ROOT = "/home/qwerty/NaturalAGI"   # ← змінити на свій шлях
SAMPLES_DIR  = os.path.join(PROJECT_ROOT, "datasets", "test")
RESULTS_DIR  = os.path.join(PROJECT_ROOT, "training_results")

# ── Інфраструктура ─────────────────────────────────────────────────────────
KAFKA_BOOTSTRAP_SERVERS = "localhost:29092"
NEO4J_URI               = "bolt://localhost:7687"
NEO4J_USER              = "neo4j"
NEO4J_PASSWORD          = "111122223333"

# ── Класи для навчання та тестування ───────────────────────────────────────
CLASSES_TO_SUBCLASSES = {
    1: [1, 3],
    2: [1, 2],
    3: [1],
    6: [1],
    7: [1],
    9: [2],
}

# ── Параметри скелетизації (experiments) ───────────────────────────────────
SKEL_CLASS_NUMBER      = 2       # цифра для демо скелетизації
SKEL_IMG_NUMBER        = 16      # номер зображення
GNG_NEURONS            = 35      # кількість нейронів GNG
GNG_ITERATIONS         = 50
RDP_EPSILON            = 5       # спрощення мережі
USE_OTSU               = True    # True = автоматичний поріг, False = FIXED_THRESHOLD
FIXED_THRESHOLD        = 180
MIN_SIZE               = 10      # видалити об'єкти менші за N пікселів
CLOSING_SIZE           = 0       # морфологічне closing (0 = вимкнено)
USE_ADAPTIVE_PRUNING   = True
PRUNE_PERCENT          = 0.05
MIN_BRANCH_LEN         = 7
MIN_ABSOLUTE           = 5

# ── Параметри класифікації ──────────────────────────────────────────────────
CLASSIFICATION_PARAMS = {
    "ged_timeout": 60,
    "skeletonization_threshold": 180,
}
SAMPLE_FRACTION = .1   # 1.0 = всі дані, 0.1 = 10% (для швидкого тесту)

print("✅ Конфігурація завантажена")
print(f"   PROJECT_ROOT: {PROJECT_ROOT}")
print(f"   Kafka:  {KAFKA_BOOTSTRAP_SERVERS}")
print(f"   Neo4j:  {NEO4J_URI}")
print(f"   Класи:  {list(CLASSES_TO_SUBCLASSES.keys())}")

## 📦 Імпорти

In [ ]:
import sys

for path in [PROJECT_ROOT, os.path.join(PROJECT_ROOT, "src")]:
    if path not in sys.path:
        sys.path.insert(0, path)

# Скелетизація
from skeletonization.settings import Settings
from skeletonization.network_simplification import NetworkSimplification
from skeletonization.preprocessing import custom_skeletonize, skeleton_to_points
from skeletonization.visualization import plot_network_result
import skeletonization.gng as gng
from cv2 import imread

# Навчання
from training.infrastructure import clean_neo4j_db, delete_test_neo4j_nodes
from training.pipeline import train_mnist, retrain_concept, remove_concept
from training.classifier import classify_image
from training.evaluation import test_mnist_all

import json, uuid
print("✅ Імпорти успішні")

---
## 🔬 1. Скелетизація — демо одного зображення

Показує всі 6 етапів обробки: оригінал → бінаризація → скелет → GNG → RDP → граф.

In [ ]:
settings = Settings(
    kafka_topic="dummy", dlq_topic="dummy", kafka_bootstrap_servers="dummy",
    N=GNG_NEURONS, maxit=GNG_ITERATIONS, L=40,
    epsilon_b=0.05, epsilon_n=0.006, alpha=0.5, delta=0.995, T=50,
)

image_id   = f"mnist_{SKEL_CLASS_NUMBER}_{SKEL_IMG_NUMBER:05d}.png"
image_path = os.path.join(SAMPLES_DIR, str(SKEL_CLASS_NUMBER), image_id)
image      = imread(image_path, 0)

print(f"Обробляємо: {image_id}")

skeleton, binary, threshold_used = custom_skeletonize(
    image,
    threshold=None if USE_OTSU else FIXED_THRESHOLD,
    max_size=MIN_SIZE - 1,
    closing_size=CLOSING_SIZE,
    min_branch_len=MIN_BRANCH_LEN,
    use_adaptive=USE_ADAPTIVE_PRUNING,
    prune_percent=PRUNE_PERCENT,
    min_absolute=MIN_ABSOLUTE,
)
print(f"Поріг: {threshold_used:.1f}")

points = skeleton_to_points(skeleton)
print(f"Точок скелету: {len(points)}")

from skeletonization.settings import gng_parameters
network = gng.fit(points, settings)
simplified = NetworkSimplification.simplify_network(network, epsilon=RDP_EPSILON)

G = plot_network_result(
    image_path, network, gng_parameters(settings),
    points, image, binary, skeleton, simplified,
    threshold_value=threshold_used,
)
print(f"Граф: {G.number_of_nodes()} вузлів, {G.number_of_edges()} ребер")

---
## 🏋️ 2. Навчання концептів

In [ ]:
# Очищення бази перед навчанням
clean_neo4j_db(uri=NEO4J_URI, user=NEO4J_USER, password=NEO4J_PASSWORD)

In [ ]:
# Навчання всіх класів з підкласами
for class_num, subclasses in CLASSES_TO_SUBCLASSES.items():
    for subclass in subclasses:
        print(f"\nНавчання: клас {class_num}, підклас {subclass}")
        train_mnist(
            class_number=class_num,
            subclass=subclass,
            is_prepared_samples=True,
            with_post_process=True,
            kafka_bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
        )

---
## 📊 3. Тестування (batch)

In [ ]:
results, y_true, y_pred = test_mnist_all(
    classes=list(CLASSES_TO_SUBCLASSES.keys()),
    params=CLASSIFICATION_PARAMS,
    sample_fraction=SAMPLE_FRACTION,
    results_dir=RESULTS_DIR,
    local_path_template=os.path.join(SAMPLES_DIR, "{cls}"),
    kafka_bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
)

---
## 🔍 4. Класифікація одного зображення

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# ── Параметри ──────────────────────────────────────────────────────────────
SINGLE_CLASS  = 3
SINGLE_IMG_ID = 422

local_path  = os.path.join(SAMPLES_DIR, str(SINGLE_CLASS))
nuclio_path = f"/opt/nuclio/shared_storage/test/{SINGLE_CLASS}"
image_name  = f"mnist_{SINGLE_CLASS}_{SINGLE_IMG_ID:05d}"

# Показати зображення
img_path = os.path.join(local_path, f"{image_name}.png")
if os.path.exists(img_path):
    plt.figure(figsize=(4, 4))
    plt.imshow(mpimg.imread(img_path), cmap="gray")
    plt.title(f"Клас {SINGLE_CLASS}, зображення #{SINGLE_IMG_ID}")
    plt.axis("off")
    plt.show()

# Класифікувати
delete_test_neo4j_nodes(uri=NEO4J_URI, user=NEO4J_USER, password=NEO4J_PASSWORD)

single_params = {
    **CLASSIFICATION_PARAMS,
    "ged_timeout": 3,
    "concept_of_interest": str(SINGLE_CLASS),
    "session_id": "test",
    "image_id": str(uuid.uuid4()),
    "delete_image_nodes": False,
}

result = classify_image(
    os.path.join(nuclio_path, f"{image_name}.png"),
    params=single_params,
    timeout=60,
    kafka_bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
)

print(json.dumps(result, indent=2))

if result["status"] == "success" and result.get("classification_results"):
    print(f"\nАктивований клас: {result['classification_results'][0]['concept_id']}")
else:
    print(f"Класифікація невдала: {result.get('error', 'unknown error')}")

---
## 🔧 Утиліти

In [ ]:
# Перенавчити один концепт (якщо щось пішло не так)
# retrain_concept(number=3, subclass=1, with_post_process=True,
#                 uri=NEO4J_URI, user=NEO4J_USER, password=NEO4J_PASSWORD)

# Видалити концепт
# remove_concept("3_1", uri=NEO4J_URI, user=NEO4J_USER, password=NEO4J_PASSWORD)